In [ ]:
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import plotly.express as px
from sklearn.feature_extraction.text import CountVectorizer
import warnings
warnings.filterwarnings("ignore")

print(" Libraries loaded")

In [ ]:
# Load Week 2 ka cleaned data
df = pd.read_csv("../data/cleaned/cleaned_speeches.csv")
print(f"Total speeches loaded: {len(df)}")
df.head()

In [ ]:
# Multilingual model (proposal mein XLM-RoBERTa suggested tha)
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# BERTopic model
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=CountVectorizer(stop_words="english"),
    min_topic_size=10,
    language="multilingual"
)

# Model fit karo (documents = cleaned text)
topics, probs = topic_model.fit_transform(df['lemmatized'].tolist())

print(" BERTopic training complete!")
print(f"Total topics found: {len(set(topics)) - 1}")  # -1 for outliers

In [ ]:
# Top 10 topics
topic_model.get_topic_info().head(10)

In [ ]:
# Add year column (agar date hai to use karo, warna simple)
df['topic'] = topics
fig = px.histogram(df, x="year", color="topic", title="Topic Evolution Over Years")
fig.show()

In [ ]:
topic_model.visualize_topics()

In [ ]:
from transformers import pipeline

# Zero-shot classifier
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

candidate_labels = ["anti-elitism", "people-centrism", "exclusionism", "neutral"]

def get_subtype_score(text):
    result = classifier(text[:512], candidate_labels)
    return result['labels'][0], round(result['scores'][0], 3)

# Test on 50 speeches
sample = df.sample(50)
sample['subtype'], sample['score'] = zip(*sample['lemmatized'].apply(get_subtype_score))

print(sample[['lemmatized', 'subtype', 'score']].head(10))